In [0]:
input_file = "dbfs:/FileStore/flight-data/json/2015_summary.json"

df1 = spark.read.json(input_file)

display(df1)

In [0]:
df1.printSchema()

**select**

In [0]:
from pyspark.sql.functions import col, column, expr, lit

**where / filter**

**orderBy / sort**

In [0]:
from pyspark.sql.functions import asc, desc

**groupBy**

- Returns a 'pyspark.sql.group.GroupedData' object (not a DataFrame)
- Apply aggregation methods to return a DataFrame

**selectExpr**

**withColumn** & **withColumnRenamed**

**drop**
- used to exclude columns in the output dataframe

In [0]:
df2.printSchema()

**dropDuplicates**

- drops duplicate rows/data

In [0]:
listUsers = [(1, "Raju", 5),
             (1, "Raju", 5),
             (3, "Raju", 5),
             (4, "Raghu", 35),
             (4, "Raghu", 35),
             (6, "Raghu", 35),
             (7, "Ravi", 70)]

users_df = spark.createDataFrame(listUsers, ["id", "name", "age"])
display(users_df)

**distinct**

In [0]:
listUsers = [(1, "Raju", 5),
             (1, "Raju", 5),
             (3, "Raju", 5),
             (4, "Raghu", 35),
             (4, "Raghu", 35),
             (6, "Raghu", 35),
             (7, "Ravi", 70)]

users_df = spark.createDataFrame(listUsers, ["id", "name", "age"])
display(users_df)

**union, intersect, subtract**

**repartition**
- Is used to increase or decrease the number of partitions of the output DF
- Causes global shuffle

**coalesce**
- Is used to only decrease the number of partitions of the output DF
- Causes partition merging

**Window functions**

In [0]:
data_file = "dbfs:/FileStore/data/empdata.csv"

In [0]:
csv_schema = "id INT, name STRING, dept STRING, salary INT"

In [0]:
windows_df = spark.read.csv(data_file, schema=csv_schema)

display(windows_df)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, desc, sum, max, avg, col, dense_rank, rank, row_number, to_date, round

In [0]:
window_spec = Window.partitionBy("dept")

In [0]:
window_df_2 = windows_df.withColumn("total_dept_salary", sum(col("salary")).over(window_spec)) \
                .withColumn("avg_dept_salary", round(avg(col("salary")).over(window_spec), 1)) \
                .withColumn("max_dept_salary", max(col("salary")).over(window_spec))

In [0]:
display(window_df_2)

In [0]:
window_spec_3 = Window \
    .partitionBy("dept") \
    .orderBy(col("salary")) \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

In [0]:
window_df_3 = windows_df.withColumn("total_salary", sum(col("salary")).over(window_spec_3)) \
                .withColumn("avg_salary", round(avg(col("salary")).over(window_spec_3), 1)) \
                .withColumn("rank", rank().over(window_spec_3)) \
                .withColumn("drank", dense_rank().over(window_spec_3)) \
                .withColumn("row_num", row_number().over(window_spec_3))

In [0]:
display(window_df_3)

**Get top 3 employees with highest salary in each department**

In [0]:
window_spec = Window.partitionBy("dept").orderBy(desc("salary"))

In [0]:
top_emp_df = windows_df.withColumn("row_num", row_number().over(window_spec)) \
                .where("row_num <= 3") \
                .drop("row_num")

In [0]:
display(top_emp_df)